In [ ]:
%reload_ext autoreload
%autoreload 2
%matplotlib inline
from collections import defaultdict
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import seaborn as sns
import pandas as pd
from pathlib import Path
import tbparse

EXPERIMENT_NAME = "tune_methods_glue_roberta"
EXPERIMENT_DIR = Path("..") / "output" / EXPERIMENT_NAME
PLOT_DIR = Path("..") / "plots"
PLOT_DIR.mkdir(exist_ok=True)
PLOT_SUFFIX = ".pdf"

TASKS = ["mnli", "qnli", "sst2", "cola", "stsb", "qqp"]
METRICS = {
    "mnli": ["eval/accuracy", "eval/loss", "train/loss"],
    "sst2": ["eval/accuracy", "eval/loss", "train/loss"],
    "qqp": ["eval/accuracy", "eval/loss", "train/loss"],
    "qnli": ["eval/accuracy", "eval/loss", "train/loss"],
    "cola": ["eval/matthews_correlation", "eval/loss", "train/loss"],
    "stsb": ["eval/spearmanr", "eval/loss", "train/loss"],
}

In [ ]:
def get_task_dfs(dedup="last"):
    task_dfs = defaultdict(list)
    for file in EXPERIMENT_DIR.glob("*"):
        run_dir = Path(file)
        if not run_dir.is_dir():
            continue  # skip non-directories
        if run_dir.name.startswith("."):
            continue  # skip hidden directories
        print(run_dir)

        df = tbparse.SummaryReader(run_dir, pivot=False).scalars
        if dedup == "first":
            df = df.drop_duplicates(subset=["step", "tag"], keep="first")
        elif dedup == "last":
            df = df.drop_duplicates(subset=["step", "tag"], keep="last")
        elif dedup == "mean":
            df = df.groupby(["step", "tag"], as_index=False)["value"].mean()
        elif dedup == "max":
            df = df.groupby(["step", "tag"], as_index=False)["value"].max()
        else:
            raise ValueError(f"Unknown dedup method: {dedup}")

        df = df.pivot(index="step", columns="tag", values="value").reset_index()

        keyvals = file.name.split(",")
        df["Seed"] = int(keyvals[0].replace("seed=", ""))
        # df["Task"] = keyvals[1].replace("task=", "")
        task = keyvals[1].replace("task=", "")
        df["Method"] = keyvals[2].replace("method=", "")
        rank = float(keyvals[3].replace("rank=", ""))
        # df["Rank"] = float(keyvals[3].replace("rank=", ""))
        df["LR"] = float(keyvals[4].replace("lr=", ""))

        task_dfs[(task, rank)].append(df)

    return {task: pd.concat(dfs_list).reset_index() for task, dfs_list in task_dfs.items()}

task_df_dict = get_task_dfs()

In [ ]:
display(next(iter(task_df_dict.values())).columns.tolist())
for (task, rank), df in task_df_dict.items():
    print(f"Task: {task}, Rank: {rank}")

In [ ]:
eval_df_dict = {}
best_lrs_dict = {}

for (task, rank), df in task_df_dict.items():
    eval_metrics = [col for col in df.columns if col.startswith("eval/")]
    main_metric = METRICS.get(task, ["eval/accuracy"])[0]
    if not eval_metrics:
        continue
    # Get df of eval metrics
    eval_df = df[["step", "Method", "LR"] + eval_metrics].dropna()
    # Average over seeds
    eval_df = eval_df.groupby(["step", "Method", "LR"], as_index=False).mean()
    # Get the best (averaged) eval accuracy over steps for each setting
    eval_df = eval_df.loc[eval_df.groupby(["Method", "LR"])[main_metric].idxmax()].reset_index(drop=True)
    # Get the best lr for each method by the max "best (averaged) eval accuracy"
    best_rows = (
        eval_df
        .groupby("Method")[["LR", main_metric]]
        .apply(lambda x: x.loc[x[main_metric].idxmax()])
        .sort_values(by=main_metric, ascending=False)
        .reset_index()
    )
    eval_df_dict[(task, rank)] = eval_df
    best_lrs_dict[(task, rank)] = best_rows


In [ ]:
# TASKS = ["mnli", "sst2", "qnli", "cola", "stsb"]
for task in TASKS:
    for rank in [8]:
        print(f"Task: {task}, Rank: {rank}")
        display(best_lrs_dict[(task, rank)])
        # display(eval_df_dict[(task, rank)])

In [ ]:
method_to_pretty = {
    "full": "Full",
    "lora": "LoRA",
    "svdlora": "SVDLoRA",
    "precond_lora": "Precond-LoRA",
    "oplora": "OPLoRA",
    "oplora_proj": "LoRA-proj",
    "oplora_scaled": "ScaledOPLoRA",
}

ranks = [8]
tasks = TASKS
fig, ax = plt.subplots(
     len(ranks), len(tasks),
     figsize=(2 + 3 * len(tasks), 2 + 3 * len(ranks)),
     squeeze=False
)
for i, rank in enumerate(ranks):
    for j, task in enumerate(tasks):
        plot_df = task_df_dict[(task, rank)].copy()
        metric_name = METRICS[task][0]

        # For each task method rank triple, select only the best lr
        plot_df = plot_df.merge(
            best_lrs_dict[(task, rank)][["Method", "LR"]],
            on=["Method", "LR"],
            how="inner",
        )

        plot_df["Method ($\\eta$)"] = plot_df.apply(
            lambda row: f"{method_to_pretty[row['Method']]} ($\\eta$={row['LR']})",
            axis=1
        )

        ax_ij = ax[i, j]
        ax_ij.grid(True, which="both", linestyle="--", linewidth=0.5)
        ax_ij.set_title(f"{task.upper()}, rank={int(rank)}")
        ax_ij.set_ylabel(metric_name)
        ax_ij.set_xlabel("Training Steps")
        # In this df, each method must have only one LR (the best one)
        lr_by_method = plot_df.groupby("Method")["LR"]
        assert lr_by_method.unique().apply(len).max() == 1, (
            "Each method should have only one LR in the filtered df"
        )
        method_lrs = lr_by_method.first().to_dict()
        # hue order should be ordered according to method_to_pretty, augmented with the lr
        hue_order = [
            f"{method_to_pretty[method]} ($\\eta$={method_lrs[method] if method in method_lrs else 'NA'})"
            for method in method_to_pretty.keys()
        ]
        sns.lineplot(
            ax=ax_ij,
            data=plot_df,
            x="step",
            y=metric_name,
            hue="Method ($\\eta$)",
            hue_order=hue_order,
            palette="tab10",
            errorbar="sd",
        )
        ax_ij.legend(fontsize=7, title="Method ($\\eta$)", loc="lower right")

# plt.suptitle("RoBERTa-base on MNLI")

# plt.savefig(PLOT_DIR / (EXPERIMENT_NAME + PLOT_SUFFIX), bbox_inches="tight")

plt.show()


In [ ]:
method_to_pretty = {
    "full": "Full",
    "lora": "LoRA",
    "svdlora": "SVDLoRA",
    "precond_lora": "Precond-LoRA",
    "oplora": "OPLoRA",
    "oplora_proj": "LoRA-proj",
    "oplora_scaled": "ScaledOPLoRA",
}

ranks = [8]
tasks = TASKS
fig, ax = plt.subplots(
     len(ranks), len(tasks),
     figsize=(2 + 3 * len(tasks), 2 + 3 * len(ranks)),
     squeeze=False
)
for i, rank in enumerate(ranks):
    for j, task in enumerate(tasks):
        plot_df = task_df_dict[(task, rank)].copy()
        metric_name = METRICS[task][0]

        # # For each task method rank triple, select only the best lr
        # plot_df = plot_df.merge(
        #     best_lrs_dict[(task, rank)][["Method", "LR"]],
        #     on=["Method", "LR"],
        #     how="inner",
        # )

        plot_df["Method ($\\eta$)"] = plot_df.apply(
            lambda row: f"{method_to_pretty[row['Method']]} ($\\eta$={row['LR']})",
            axis=1
        )

        ax_ij = ax[i, j]
        ax_ij.grid(True, which="both", linestyle="--", linewidth=0.5)
        ax_ij.set_title(f"{task.upper()}, rank={int(rank)}")
        ax_ij.set_ylabel(metric_name)
        ax_ij.set_xlabel("Training Steps")
        sns.lineplot(
            ax=ax_ij,
            data=plot_df,
            x="step",
            y=metric_name,
            hue="Method ($\\eta$)",
            # hue_order=list(method_to_pretty.values()),
            # style="LR",
            palette="tab10",
            errorbar="sd",
        )
        ax_ij.legend(fontsize=7, title="Method ($\\eta$)", loc="lower right")

# plt.suptitle("RoBERTa-base on MNLI")

# plt.savefig(PLOT_DIR / (EXPERIMENT_NAME + PLOT_SUFFIX), bbox_inches="tight")

plt.show()


In [ ]:
# TASKS = ["mnli", "sst2", "qnli", "cola", "stsb"]
for task in TASKS:
    for rank in [8]:
        print(f"Task: {task}, Rank: {rank}")
        # display(best_lrs_dict[(task, rank)])
        display(eval_df_dict[(task, rank)])